# 03 · Análisis exploratorio y ficha de variables
Lee **solo** `data/processed/`. Cada hallazgo indica su cifra, su *n* y la figura en `figs/`.

**Salidas:** `figs/*.png`, `results/hallazgos.json`, `results/prueba_hipotesis_datos.json`,
`data/processed/ficha_candidatas.csv`.

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / 'src'))
import numpy as np, pandas as pd
import proyecto as P
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
print('Raíz del proyecto:', P.RAIZ)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

df = P.leer_tabla('dataset_final')
y = df[P.OBJETIVO]
H = {}   # hallazgos numéricos -> results/hallazgos.json
print(f'n = {len(df):,} | tasa global = {y.mean()*100:.2f}%')

## Hallazgo 1 · Desbalance de la variable objetivo
La clase positiva (reprobado o retirado) es minoritaria: la exactitud (*accuracy*) no sirve como métrica principal.

In [ ]:
H['h1'] = {'n': int(len(df)), 'positivos': int(y.sum()), 'tasa_pct': float(y.mean() * 100)}
por_anio = df.groupby('AGNO')[P.OBJETIVO].agg(n='size', positivos='sum', tasa_pct=lambda s: s.mean() * 100)
display(por_anio.round(2))
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(['Promovido (0)', 'Reprobado/Retirado (1)'], [(y == 0).mean() * 100, (y == 1).mean() * 100], color=['#9db4d6', '#c0504d'])
ax.set_ylabel('% de estudiantes'); ax.set_title('Distribución del objetivo')
fig.savefig(P.DIR_FIGS / '01_dist_objetivo.png', bbox_inches='tight'); plt.show()

## Hallazgo 2 · El rendimiento del año anterior (t−1) frente al objetivo del año t
Se compara la tasa de fracaso según el promedio de t−1: **sin dato**, **≥ 4,0** y **< 4,0**. Un promedio bajo 4,0
aumenta el riesgo, pero **no lo determina**: la mayoría de los estudiantes con promedio bajo 4,0 igual es promovida.

In [ ]:
cat = np.where(df['prom_prev_faltante'] == 1, '1. sin promedio en t-1',
               np.where(df['riesgo_academico_prev'] == 1, '3. promedio < 4,0', '2. promedio >= 4,0'))
tabla2 = df.groupby(cat)[P.OBJETIVO].agg(n='size', fracasos='sum', tasa_pct=lambda s: s.mean() * 100).round(2)
display(tabla2)
valido = df['prom_prev_faltante'] == 0
rr = P.razon_de_riesgo(y[valido], df.loc[valido, 'riesgo_academico_prev'])
H['h2'] = {'tabla': tabla2.reset_index().rename(columns={'index': 'grupo'}).to_dict('records'), 'razon_de_riesgo': rr}
print('Razón de riesgo (prom<4 vs >=4):', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in rr.items()})
print('=> entre quienes tuvieron promedio < 4,0 en t-1, %.1f%% reprueba/abandona y %.1f%% es promovido.' % (rr['tasa_expuestos'] * 100, (1 - rr['tasa_expuestos']) * 100))

m = df[valido].sample(min(300_000, int(valido.sum())), random_state=P.SEMILLA)
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([m.loc[m[P.OBJETIVO] == 0, 'prom_notas_norm_prev'], m.loc[m[P.OBJETIVO] == 1, 'prom_notas_norm_prev']],
           tick_labels=['Promovido (0)', 'Reprobado/Retirado (1)'], showfliers=False)
ax.set_ylabel('Promedio normalizado en t-1'); ax.set_title('Promedio del año anterior según objetivo en t')
fig.savefig(P.DIR_FIGS / '02_prom_prev_vs_objetivo.png', bbox_inches='tight'); plt.show()

## Hallazgo 3 · Notas faltantes o iguales a 0,0 en t−1
Un 0,0 es ausencia de dato, no una nota. Se trata como **faltante con bandera aparte** (`prom_prev_faltante`) y no
como riesgo académico. Se muestra en qué niveles de enseñanza se concentra y qué tasa de fracaso tienen.

In [ ]:
f = df.groupby('COD_ENSE2').agg(n=('MRUN', 'size'), pct_sin_nota=('prom_prev_faltante', lambda s: s.mean() * 100),
                                tasa_fracaso_pct=(P.OBJETIVO, lambda s: s.mean() * 100)).round(2)
f['aporte_a_faltantes_pct'] = (df.groupby('COD_ENSE2')['prom_prev_faltante'].sum() / df['prom_prev_faltante'].sum() * 100).round(1)
display(f.sort_values('pct_sin_nota', ascending=False))
H['h3'] = {'pct_sin_prom_prev': float(df['prom_prev_faltante'].mean() * 100),
           'tasa_fracaso_sin_nota': float(y[df['prom_prev_faltante'] == 1].mean() * 100),
           'tasa_fracaso_con_nota': float(y[df['prom_prev_faltante'] == 0].mean() * 100),
           'por_nivel': f.reset_index().to_dict('records')}
fig, ax = plt.subplots(figsize=(6, 3.5))
f['pct_sin_nota'].plot.bar(ax=ax, color='#7b9ac7'); ax.set_ylabel('% sin promedio en t-1'); ax.set_xlabel('COD_ENSE2 (nivel)')
ax.set_title('Faltantes de nota previa por nivel de enseñanza')
fig.savefig(P.DIR_FIGS / '03_faltantes_nivel.png', bbox_inches='tight'); plt.show()

## Hallazgo 4 · Segmentación por nivel de enseñanza

In [ ]:
s4 = df.groupby('COD_ENSE2').agg(n=('MRUN', 'size'), tasa_pct=(P.OBJETIVO, lambda s: s.mean() * 100)).round(2)
display(s4)
grandes = s4[s4['n'] >= 5000]
rango = float(grandes['tasa_pct'].max() - grandes['tasa_pct'].min())
H['h4'] = {'rango_pp_entre_niveles_con_n>=5000': rango, 'tabla': s4.reset_index().to_dict('records')}
print(f'Diferencia máxima de tasa entre niveles (n>=5000): {rango:.1f} puntos porcentuales')
fig, ax = plt.subplots(figsize=(6, 3.5))
s4['tasa_pct'].plot.bar(ax=ax, color='#c0504d'); ax.set_ylabel('% reprobado/retirado'); ax.set_xlabel('COD_ENSE2 (nivel)')
ax.set_title('Tasa de fracaso por nivel')
fig.savefig(P.DIR_FIGS / '04_seg_nivel.png', bbox_inches='tight'); plt.show()

## Hallazgo 5 · Estabilidad entre años (y composición)
Si la tasa y la composición cambian mucho entre cohortes, una partición temporal (entrenar en el pasado, probar
en el último año) es más exigente y más realista que una partición aleatoria.

In [ ]:
comp = pd.crosstab(df['AGNO'], df['COD_ENSE2'], normalize='index').mul(100).round(1)
display(por_anio.round(2)); display(comp)
tasas = por_anio['tasa_pct']
H['h5'] = {'tasa_por_anio': tasas.to_dict(), 'dif_pp_entre_anios': float(tasas.max() - tasas.min()),
           'composicion_nivel_pct': comp.to_dict()}
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(tasas.index.astype(str), tasas.values, marker='o', color='#c0504d'); ax.set_ylabel('% reprobado/retirado')
ax.set_title('Tasa por cohorte (año t)')
fig.savefig(P.DIR_FIGS / '05_evolucion_anio.png', bbox_inches='tight'); plt.show()

## Hallazgo 6 · Asistencia del año anterior
Reemplaza al antiguo hallazgo de colinealidad (r = 1 por construcción, trivial). La asistencia de t−1 es una
señal distinta del promedio: se mide su relación con el objetivo y con el promedio.

In [ ]:
tramos = pd.cut(df['asistencia_prev'], [0, 60, 80, 90, 95, 100.01], labels=['<=60', '60-80', '80-90', '90-95', '95-100'], include_lowest=True)
t6 = df.groupby(tramos, observed=True)[P.OBJETIVO].agg(n='size', tasa_pct=lambda s: s.mean() * 100).round(2)
display(t6)
ok = df['asistencia_prev'].notna() & df['prom_notas_norm_prev'].notna()
rho = df.loc[ok, ['asistencia_prev', 'prom_notas_norm_prev']].sample(min(300_000, int(ok.sum())), random_state=P.SEMILLA).corr(method='spearman').iloc[0, 1]
H['h6'] = {'tabla': t6.reset_index().astype({'asistencia_prev': str}).to_dict('records'), 'spearman_asistencia_vs_promedio_prev': float(rho),
           'pct_sin_asistencia_prev': float(df['asistencia_prev_faltante'].mean() * 100)}
print('Spearman asistencia_prev vs promedio_prev: %.2f' % rho)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
t6['tasa_pct'].plot.bar(ax=ax, color='#7b9ac7'); ax.set_ylabel('% reprobado/retirado'); ax.set_xlabel('Asistencia en t-1 (%)')
ax.set_title('Fracaso según asistencia previa')
fig.savefig(P.DIR_FIGS / '06_asistencia_tramos.png', bbox_inches='tight'); plt.show()

## Hallazgo 7 · Cobertura: quién queda fuera por exigir el año previo (t−1)
El cruce exige registro en t−1, por lo que **no hay faltantes en t−1 por construcción** y quedan fuera los
ingresantes al sistema. Se cuantifica cuántos son y cómo se comportan.

In [ ]:
cob = pd.read_csv(P.DIR_RES / 'cobertura.csv')
cols = ['anio', 'con_objetivo', 'sin_registro_en_t_menos_1', 'dataset_final', 'tasa_objetivo_dataset_final', 'tasa_objetivo_sin_t_menos_1']
display(cob[cols])
H['h7'] = cob[cols].to_dict('records')

## Hipótesis (reformuladas como afirmaciones sobre los datos)
Se contrastan aquí la H1 (datos) y en el cuaderno 04 las H2 y H3 (modelo, solo con validación cruzada del entrenamiento).

* **H1.** La tasa de reprobación/retiro en *t* es mayor entre quienes tuvieron promedio < 4,0 en *t−1* que entre
  quienes tuvieron ≥ 4,0 (razón de riesgo > 1).
* **H2.** Agregar el nivel de enseñanza (`COD_ENSE`) al historial de notas mejora la capacidad de ordenar el riesgo
  (PR-AUC en validación cruzada) respecto de usar solo el promedio previo. *(cuaderno 04)*
* **H3.** Tratar el 0,0/ausencia de nota como faltante con bandera (y no como riesgo = 1) reduce los falsos
  positivos entre estudiantes sin nota previa. *(cuaderno 04)*

In [ ]:
rr_h1 = P.razon_de_riesgo(y[valido], df.loc[valido, 'riesgo_academico_prev'])
rr_rep = P.razon_de_riesgo(y, df['repro_prev'])
P.guardar_json({'H1_promedio_menor_4': rr_h1, 'extra_repitio_en_t_menos_1': rr_rep}, 'prueba_hipotesis_datos.json')
print('H1: RR = %.2f (IC95%% %.2f-%.2f), p = %.3g -> %s' % (rr_h1['RR'], *rr_h1['IC95'], rr_h1['p_valor'],
      'se respalda' if rr_h1['IC95'][0] > 1 else 'NO se respalda'))
print('Repitió en t-1: RR = %.2f' % rr_rep['RR'])

## Ficha de variables candidatas
Calculada sobre los datos (no escrita a mano): relación con el objetivo, si **sobrevive a la segmentación**
(misma dirección y al menos la mitad de la fuerza global en cada estrato grande de año × nivel/género),
redundancia entre variables (|ρ de Spearman| > 0,9) y decisión. La columna *% faltante* es 0 por construcción
para las variables de t−1 *entre quienes tienen registro previo* (ver hallazgo 7).

In [ ]:
ficha = P.ficha_candidatas(df)
ficha.to_csv(P.DIR_PROC / 'ficha_candidatas.csv', index=False)
display(ficha)
print('Entran al modelo:', ficha.loc[ficha['decision'] == 'Entra', 'variable'].tolist())

### Fuga demostrada (del cuaderno 02)
Esta tabla justifica dejar **fuera** el promedio del año *t*.

In [ ]:
import json
fuga = json.load(open(P.DIR_RES / 'auditoria_fuga.json'))
print(fuga)
P.guardar_json(H, 'hallazgos.json')